# AutoML quick guide with autolog enabled

## Introduction
In this notebook, you'll see how to perform an AutoML task with FLAML for different scenarios. We'll have mlflow autolog enabled, so you'll also see how AutoML integrated with mlflow.

The scenarios are as below:

1. Pandas dataframe as input

    In this scenario, we have a dataset in pandas dataframe format, and we'll perform a regression task. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default, while the run names will be randomly generated words.

2. Spark dataframe as input

    In this scenario, we have the same dataset but in spark dataframe format. And we'll customize mlflow experiment name and run name.

Please ref [FLAML doc](https://microsoft.github.io/FLAML/docs/Getting-Started/) for more details of FLAML usage.

## Prerequisites
We need to install flaml for performing automl tasks.

In [ ]:
%pip install "flaml[synapse]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl"

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, -1, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 1.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 3.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 46.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.3/212.3 kB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 45.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 29.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 40.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 37.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0 -> 23.1.1
[notice] To update, run: /nfs4/pyenv-f95eb1d7-acd8-48ef-be6c-b3d18e2c82ca/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Case 1. Pandas dataframe as input

In this scenario, we have a dataset in pandas dataframe format, and we'll perform a regression task. For the mlflow integration, we'll not set mlflow experiment name and run name; the experiment name will be the notebook name by default, while the run names will be randomly generated words.

![automl_exp_1.png](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/automl_autolog_on_1.png)

In [ ]:
import time
import mlflow
import flaml
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, 9, Finished, Available)

In [ ]:
automl_experiment = flaml.AutoML()
automl_settings = {
    "max_iter": 3,
    "metric": "r2",
    "task": "regression",
    "n_concurrent_trials": 2,
    "use_spark": True,  # use spark to parallelize the training
    "log_type": "better",  # flaml only logs better configs than the previous iters by default, set to "all" to log all trials
}
X, y = load_diabetes(return_X_y=True, as_frame=True)
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25)
automl_experiment.fit(X_train=train_x, y_train=train_y, **automl_settings)

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, 10, Finished, Available)

[flaml.automl.logger: 04-24 11:23:50] {1699} INFO - task = regression
[flaml.automl.logger: 04-24 11:23:50] {1706} INFO - Data split method: uniform
[flaml.automl.logger: 04-24 11:23:50] {1709} INFO - Evaluation method: cv
[flaml.automl.logger: 04-24 11:23:50] {1807} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 04-24 11:23:50] {1917} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth']
[flaml.tune.tune: 04-24 11:23:51] {719} INFO - Number of trials: 1/3, 1 RUNNING, 0 TERMINATED
[flaml.tune.tune: 04-24 11:24:18] {742} INFO - Brief result: {'pred_time': 4.299837558423702e-05, 'wall_clock_time': 28.205904483795166, 'metric_for_logging': {'pred_time': 4.299837558423702e-05}, 'val_loss': 0.7829718165843209, 'trained_estimator': <flaml.automl.model.LGBMEstimator object at 0x7f04e7bb50c0>}
[flaml.tune.tune: 04-24 11:24:18] {719} INFO - Number of trials: 2/3, 1 RUNNING, 1 TERMINATED
[flaml.tune.tune: 04-24 11:24:46] {742} INFO - Brie

In [ ]:
print(automl_experiment.model)
print(automl_experiment.config_history)
print(automl_experiment.best_iteration)
print(automl_experiment.best_estimator)

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, 11, Finished, Available)

{0: ('lgbm', {'ml': {'n_estimators': 4, 'num_leaves': 4, 'min_child_samples': 20, 'learning_rate': 0.09999999999999995, 'log_max_bin': 8, 'colsample_bytree': 1.0, 'reg_alpha': 0.0009765625, 'reg_lambda': 1.0, 'learner': 'lgbm'}}, 0), 1: ('rf', {'ml': {'n_estimators': 4, 'max_features': 1.0, 'max_leaves': 4, 'learner': 'rf'}}, 28.205904483795166)}
1
rf


## Case 2. Spark dataframe as input

In this scenario, we have the same dataset but in spark dataframe format. And we'll customize mlflow experiment name and run name.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/demo-images/automl_exp_1.png)

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from flaml.automl.spark.utils import to_pandas_on_spark

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, 12, Finished, Available)

In [ ]:
pd_df = load_diabetes(as_frame=True).frame
df = spark.createDataFrame(pd_df)
df = df.repartition(4).cache()
df.count()
train, test = df.randomSplit([0.8, 0.2], seed=1)
feature_cols = df.columns[:-1]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train)["target", "features"]
test_data = featurizer.transform(test)["target", "features"]
automl = flaml.AutoML()
# no need to set use_spark since a spark model itself will run in parallel
settings = {
    "max_iter": 3,
    "metric": "mse",
    "task": "regression",  # task type
    "seed": 7654321,  # random seed
}
df = to_pandas_on_spark(train_data)

mlflow.set_experiment("automl_exp")
with mlflow.start_run(nested=True, run_name="automl_run"):
    automl.fit(dataframe=df, label="target", **settings)

model = automl.model.estimator
predictions = model.transform(test_data)
predictions.show(10)

evaluator = RegressionEvaluator(
    labelCol="target", predictionCol="prediction", metricName="mse"
)
metric = evaluator.evaluate(predictions)
print(f"mse: {metric}")

StatementMeta(, 4aeb92b0-fa7a-42e0-a40b-29b5df7d8a54, 13, Finished, Available)

[flaml.automl.logger: 04-24 11:25:22] {1699} INFO - task = regression
[flaml.automl.logger: 04-24 11:25:22] {1706} INFO - Data split method: uniform
[flaml.automl.logger: 04-24 11:25:22] {1709} INFO - Evaluation method: cv
[flaml.automl.logger: 04-24 11:25:22] {1807} INFO - Minimizing error metric: mse
[flaml.automl.logger: 04-24 11:25:22] {1917} INFO - List of ML learners in AutoML Run: ['lgbm_spark']
[flaml.automl.logger: 04-24 11:25:22] {2209} INFO - iteration 0, current learner lgbm_spark
[flaml.automl.logger: 04-24 11:26:07] {2337} INFO - Estimated sufficient time budget=10000s. Estimated necessary time budget=10s.
[flaml.automl.logger: 04-24 11:26:07] {2386} INFO -  at 47.1s,	estimator lgbm_spark's best error=4653.9237,	best estimator lgbm_spark's best error=4653.9237
[flaml.automl.logger: 04-24 11:26:07] {2209} INFO - iteration 1, current learner lgbm_spark
[flaml.automl.logger: 04-24 11:26:48] {2386} INFO -  at 87.4s,	estimator lgbm_spark's best error=4653.9237,	best estimator 